In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import glob
import datetime

## Setup

In [ ]:
def to_binary_string(num):
    """
    Turns an input integer into a binary string.

    Args:
    num (int): Number to convert to a binary string.
               Gets converted to an integer if it is not already

    Returns:
    (str): String of the binary value of the input integer
    """
    if np.isnan(num):
        return np.nan
    else:
        return f'{int(num):b}'

In [ ]:
"""
Export coordinates for each urban area
KEY: [utm zone, T/F Northern Hemisphere]
"""
proj_zone = {
    'KBWI':[18, True], 'KJFK':[18, True], 'KPHX':[12, True], 'KMIA':[17, True], 'KORD':[16, True], 'KDEN':[13, True],
    'KSEA':[10, True], 'KSFO':[10, True], 'KLAX':[11, True], 'KATL':[16, True], 'CYYZ':[17, True],
    'MMMX':[14, True], 'KLAS':[11, True], 'KSLC':[12, True], 'KDFW':[14, True], 'KIAH':[15, True],
    'KMSY':[15, True], 'KSTL':[15, True], 'KMSP':[15, True], 'KJAX':[17, True], 'KCLT':[17, True],
    'KPHL':[18, True], 'KSAN':[11, True], 'TJSJ':[19, True], 'CYUL':[18, True], 'MGGL':[13, True],
    'MMMY':[14, True], 'MMUN':[16, True], 'KBIL':[12, True], 'MGGT':[15, True], 'MROC':[16, True],
    'MUHA':[17, True], 'MDSD':[19, True], 'MHTG':[16, True], 'MNMG':[16, True], 'MPTO':[17, True],
    'SKBO':[18, True], 'SPJC':[18, False], 'SEQM':[17, True], 'SCEL':[19, False], 'SAEZ':[21, False],
    'SBGR':[23, False], 'SBEG':[20, False], 'SCCI':[19, False],
    'SLLP':[19, False], 'SUMU':[21, False], 'SBBR':[22, False], 'SVMI':[19, True]
}

# Creates a sorted list of the city names
cities = sorted(list(proj_zone.keys()))

In [ ]:
# Dictionary for full city name strings
city_str_dict = {
    'KATL': 'Atlanta, Georgia, USA',
    'KBIL': 'Billings, Montana, USA',
    'SKBO': 'Bogota, Colombia',
    'SBBR': 'Brasilia, Brazil',
    'SAEZ': 'Buenos Aires, Argentina',
    'MMUN': 'Cancun, Mexico',
    'SVMI': 'Caracas, Venezuela',
    'KCLT': 'Charlotte, North Carolina, USA',
    'KORD': 'Chicago, Illinois, USA',
    'KDFW': 'Dallas, Texas, USA and Fort Worth, Texas, USA',
    'KDEN': 'Denver, Colorado, USA',
    'MMGL': 'Guadalajara, Mexico',
    'MGGT': 'Guatemala City, Guatemala',
    'MUHA': 'Havana, Cuba',
    'KIAH': 'Houston, Texas, USA',
    'KJAX': 'Jacksonville, Florida, USA',
    'SLLP': 'La Paz, Bolivia',
    'KLAS': 'Las Vegas, Nevada, USA',
    'SPJC': 'Lima, Peru',
    'KLAX': 'Los Angeles, California, USA',
    'MNMG': 'Managua, Nicaragua',
    'SBEG': 'Manaus, Brazil',
    'MMMX': 'Mexico City, Mexico',
    'KMIA': 'Miami, Florida, USA',
    'KMSP': 'Minneapolis, Minnesota, USA',
    'MMMY': 'Monterrey, Mexico',
    'SUMU': 'Montevideo, Uruguay',
    'CYUL': 'Montreal, Quebec, Canada',
    'KMSY': 'New Orleans, Louisiana, USA',
    'KJFK': 'New York City, New York, USA',
    'MPTO': 'Panama City, Panama',
    'KPHL': 'Philadelphia, Pennsylvania, USA',
    'KPHX': 'Phoenix, Arizona, USA',
    'SCCI': 'Punta Arenas, Chile',
    'SEQM': 'Quito, Ecuador',
    'KSLC': 'Salt Lake City, Utah, USA',
    'KSAN': 'San Diego, California, USA and Tijuana, Mexico',
    'KSFO': 'San Francisco, California, USA and San Jose, California, USA',
    'MROC': 'San Jose, Costa Rica',
    'TJSJ': 'San Juan, Puerto Rico',
    'SCEL': 'Santiago, Chile',
    'MDSD': 'Santo Domingo, Dominican Republic',
    'SBGR': 'Sao Paulo, Brazil',
    'KSEA': 'Seattle, Washington, USA',
    'KSTL': 'St Louis, Missouri, USA',
    'MHTG': 'Tegucigalpa, Honduras',
    'CYYZ': 'Toronto, Ontario, Canada',
    'KBWI': 'Washington, DC, USA and Baltimore, Maryland, USA'
}

In [ ]:
tmp_dataset_root = '/glade/derecho/scratch/jstarfeldt/uhminicubes_tmp'
dataset_root = '/glade/derecho/scratch/jstarfeldt/uhminicubes'

In [ ]:
"""
Returns an integer as a str, adding a zero in front if it is a single digit.

Attributes:
    num (int): Integer to turn into a string.
"""
def stringify(num):
    if num >= 0 and num < 10:
        return f'0{num}'
    else:
        return str(num)

In [ ]:
def sort_func_Landsat(s, form=int):
    """
    Gets the datetime from a Landsat/Sentinel-1 file string.
    Used to sort the list of files in a directory.

    Args:
    s (str): Full file location string of a Landsat/Sentinel-1 file

    Returns:
    (int): File datetime as an integer
    """
    if form == int:
        return int(s.split(f'{city}_')[1].split('.tif')[0])
    else:
        return str(s.split(f'{city}_')[1].split('.tif')[0])

## Landsat processing

In [ ]:
city = 'SKBO'
# Loads the Landsat file times for the current city from GEE
times_Landsat = pd.read_csv(f'{tmp_dataset_root}/Landsat_times/Landsat_times_{city}.csv').value
times_Landsat[:5]

In [ ]:
# Gets the list of unprocessed Landsat/Sentinel-1 files for the current city
Landsat_root = f'{tmp_dataset_root}/{city}/Landsat'
Landsat_tif_list = glob.glob(Landsat_root+'/Landsat8*.tif')
Landsat_tif_list[0]

In [ ]:
# Sorts the list of Landsat/Sentinel-1 files
Landsat_tif_list = sorted(Landsat_tif_list, key=sort_func_Landsat)
Landsat_tif_list[0]

In [ ]:
def process_Landsat_Sentinel_tif(tif, time, coord_bounds=None):
    """
    Processing of individual Landsat/Sentinel-1 .tif files. Performs
    a variety of tasks on the data to make it easier to read and understand.

    Args:
        tif (str): Path where tif file is located
        time (str): Date and time of when the data was collected format YYYY-MM-DDThh:mm:ssZ.
        coord_bounds (tuple or list, optional): Coordinate bounds if you wish to filter the data by location. Order should be
                                        (longitude minimum, longitude maximum, latitude minimum, latitude maximum)

    Returns:
    geotiff_ds (xr.dataset): Fully processed file as an xarray dataset
    """
    #########################################################################################################
    # Open file and rename variables
    dsLS = rxr.open_rasterio(tif)[:,:3000,:3000] 
    dsLS = dsLS.reindex(y=dsLS.y[::-1]) # Invert y-axis so that values are increasing instead of decreasing with index
    geotiff_ds = dsLS.to_dataset('band')

    geotiff_ds = geotiff_ds.rename({1:'Landsat_blue_sfc_reflectance', 2:'Landsat_green_sfc_reflectance',
                                    3:'Landsat_red_sfc_reflectance', 4:'Landsat_NIR_sfc_reflectance',
                                    5:'Landsat_SWIR1_sfc_reflectance', 6:'Landsat_SWIR2_sfc_reflectance',
                                    7:'Landsat_LST', 8:'Landsat_cloud_mask', 9:'Sentinel1_VV', 10:'Sentinel1_VH', 11:'Sentinel1_HH',
                                    12:'Sentinel1_HV', 13:'Sentinel1_incidence_angle'})
    geotiff_ds = geotiff_ds.assign_coords({'datetime': time})
    del geotiff_ds.attrs['long_name']

    #########################################################################################################
    # Convert cloud mask integers to binary strings
    binary_vectorized = np.vectorize(to_binary_string)
    mask = binary_vectorized(geotiff_ds.Landsat_cloud_mask.values)
    geotiff_ds['Landsat_cloud_mask'].values = mask

    # Filter anomalous values from Sentinel-1 variables and set them to nan
    for variable in ['Sentinel1_VV', 'Sentinel1_VH', 'Sentinel1_HH', 'Sentinel1_HV']:
        geotiff_ds[variable].values = np.where(geotiff_ds[variable].values > 1, np.nan, geotiff_ds[variable].values)
    geotiff_ds['Sentinel1_incidence_angle'].values = np.where(geotiff_ds['Sentinel1_incidence_angle'].values < 0,
                                                      np.nan, geotiff_ds['Sentinel1_incidence_angle'].values)

    #########################################################################################################
    # Assign descriptive attributes for the cloud mask and a new coordinate for the file datetime
    if 'Landsat9' in tif:
        geotiff_ds.attrs['title'] = f'Landsat 9 and Sentinel-1 data for {city_str_dict[city]}'
    else:
        geotiff_ds.attrs['title'] = f'Landsat 8 and Sentinel-1 data for {city_str_dict[city]}'
    geotiff_ds.attrs['institution'] = 'University of Maryland, College Park'
    geotiff_ds.attrs['source'] = 'Satellite observation'

    geotiff_ds['y'].attrs['standard_name'] = 'projection_y_coordinate'
    geotiff_ds['y'].attrs['long_name'] = 'UTM Northing'
    geotiff_ds['y'].attrs['units'] = 'm'

    geotiff_ds['x'].attrs['standard_name'] = 'projection_x_coordinate'
    geotiff_ds['x'].attrs['long_name'] = 'UTM Easting'
    geotiff_ds['x'].attrs['units'] = 'm'

    geotiff_ds['datetime'].attrs['long_name'] = 'datetime'
    geotiff_ds['datetime'].attrs['units'] = 'YYYY-mm-DDTHH:MM:SSZ'
    geotiff_ds['datetime'].attrs['calendar'] = 'utc'

    geotiff_ds['Landsat_blue_sfc_reflectance'].attrs['standard_name'] = 'surface_bidirectional_reflectance'
    geotiff_ds['Landsat_blue_sfc_reflectance'].attrs['long_name'] = 'L2_blue_surface_reflectance'
    geotiff_ds['Landsat_blue_sfc_reflectance'].attrs['units'] = '1'
    geotiff_ds['Landsat_blue_sfc_reflectance'].attrs['valid_min'] = 0
    geotiff_ds['Landsat_blue_sfc_reflectance'].attrs['valid_max'] = 1
    geotiff_ds['Landsat_blue_sfc_reflectance'].attrs['missing_value'] = np.nan
    geotiff_ds['Landsat_blue_sfc_reflectance'].attrs['wavelength'] = '0.452-0.512 μm'

    geotiff_ds['Landsat_green_sfc_reflectance'].attrs['standard_name'] = 'surface_bidirectional_reflectance'
    geotiff_ds['Landsat_green_sfc_reflectance'].attrs['long_name'] = 'L2_green_surface_reflectance'
    geotiff_ds['Landsat_green_sfc_reflectance'].attrs['units'] = '1'
    geotiff_ds['Landsat_green_sfc_reflectance'].attrs['valid_min'] = 0
    geotiff_ds['Landsat_green_sfc_reflectance'].attrs['valid_max'] = 1
    geotiff_ds['Landsat_green_sfc_reflectance'].attrs['missing_value'] = np.nan
    geotiff_ds['Landsat_green_sfc_reflectance'].attrs['wavelength'] = '0.533-0.590 μm'

    geotiff_ds['Landsat_red_sfc_reflectance'].attrs['standard_name'] = 'surface_bidirectional_reflectance'
    geotiff_ds['Landsat_red_sfc_reflectance'].attrs['long_name'] = 'L2_red_surface_reflectance'
    geotiff_ds['Landsat_red_sfc_reflectance'].attrs['units'] = '1'
    geotiff_ds['Landsat_red_sfc_reflectance'].attrs['valid_min'] = 0
    geotiff_ds['Landsat_red_sfc_reflectance'].attrs['valid_max'] = 1
    geotiff_ds['Landsat_red_sfc_reflectance'].attrs['missing_value'] = np.nan
    geotiff_ds['Landsat_red_sfc_reflectance'].attrs['wavelength'] = '0.636-0.673 μm'

    geotiff_ds['Landsat_NIR_sfc_reflectance'].attrs['standard_name'] = 'surface_bidirectional_reflectance'
    geotiff_ds['Landsat_NIR_sfc_reflectance'].attrs['long_name'] = 'L2_NIR_surface_reflectance'
    geotiff_ds['Landsat_NIR_sfc_reflectance'].attrs['units'] = '1'
    geotiff_ds['Landsat_NIR_sfc_reflectance'].attrs['valid_min'] = 0
    geotiff_ds['Landsat_NIR_sfc_reflectance'].attrs['valid_max'] = 1
    geotiff_ds['Landsat_NIR_sfc_reflectance'].attrs['missing_value'] = np.nan
    geotiff_ds['Landsat_NIR_sfc_reflectance'].attrs['wavelength'] = '0.851-0.879 μm'

    geotiff_ds['Landsat_SWIR1_sfc_reflectance'].attrs['standard_name'] = 'surface_bidirectional_reflectance'
    geotiff_ds['Landsat_SWIR1_sfc_reflectance'].attrs['long_name'] = 'L2_SWIR1_surface_reflectance'
    geotiff_ds['Landsat_SWIR1_sfc_reflectance'].attrs['units'] = '1'
    geotiff_ds['Landsat_SWIR1_sfc_reflectance'].attrs['valid_min'] = 0
    geotiff_ds['Landsat_SWIR1_sfc_reflectance'].attrs['valid_max'] = 1
    geotiff_ds['Landsat_SWIR1_sfc_reflectance'].attrs['missing_value'] = np.nan
    geotiff_ds['Landsat_SWIR1_sfc_reflectance'].attrs['wavelength'] = '1.566-1.651 μm'

    geotiff_ds['Landsat_SWIR2_sfc_reflectance'].attrs['standard_name'] = 'surface_bidirectional_reflectance'
    geotiff_ds['Landsat_SWIR2_sfc_reflectance'].attrs['long_name'] = 'L2_SWIR2_surface_reflectance'
    geotiff_ds['Landsat_SWIR2_sfc_reflectance'].attrs['units'] = '1'
    geotiff_ds['Landsat_SWIR2_sfc_reflectance'].attrs['valid_min'] = 0
    geotiff_ds['Landsat_SWIR2_sfc_reflectance'].attrs['valid_max'] = 1
    geotiff_ds['Landsat_SWIR2_sfc_reflectance'].attrs['missing_value'] = np.nan
    geotiff_ds['Landsat_SWIR2_sfc_reflectance'].attrs['wavelength'] = '2.107-2.294 μm'

    geotiff_ds['Landsat_LST'].attrs['standard_name'] = 'surface_temperature'
    geotiff_ds['Landsat_LST'].attrs['long_name'] = 'L2_band_10_land_surface_temperature'
    geotiff_ds['Landsat_LST'].attrs['units'] = 'K'
    geotiff_ds['Landsat_LST'].attrs['valid_min'] = 0
    geotiff_ds['Landsat_LST'].attrs['valid_max'] = 373
    geotiff_ds['Landsat_LST'].attrs['wavelength'] = '10.60-11.19 μm'
    geotiff_ds['Landsat_LST'].attrs['missing_value'] = np.nan

    geotiff_ds['Landsat_cloud_mask'].attrs['long_name'] = 'pixel_quality_mask'
    geotiff_ds['Landsat_cloud_mask'].attrs['bitmask_key'] = \
    "Bit 0: Fill\n\
    Bit 1: Dilated Cloud\n\
    Bit 2: Cirrus (high confidence)\n\
    Bit 3: Cloud\n\
    Bit 4: Cloud Shadow\n\
    Bit 5: Snow\n\
    Bit 6: Clear\n\t0: Cloud or Dilated Cloud bits are set\n\t1: Cloud and Dilated Cloud bits are not set\n\
    Bit 7: Water\n\
    Bits 8-9: Cloud Confidence\n\t0: None\n\t1: Low\n\t2: Medium\n\t3: High\n\
    Bits 10-11: Cloud Shadow Confidence\n\t0: None\n\t1: Low\n\t2: Medium\n\t3: High\n\
    Bits 12-13: Snow/Ice Confidence\n\t0: None\n\t1: Low\n\t2: Medium\n\t3: High\n\
    Bits 14-15: Cirrus Confidence\n\t0: None\n\t1: Low\n\t2: Medium\n\t3: High"

    geotiff_ds['Sentinel1_VV'].attrs['standard_name'] = 'surface_backwards_scattering_coefficient_of_radar_wave'
    geotiff_ds['Sentinel1_VV'].attrs['long_name'] = 'SAR_VV_polarized_backscatter'
    geotiff_ds['Sentinel1_VV'].attrs['units'] = 'dB'
    geotiff_ds['Sentinel1_VV'].attrs['valid_max'] = 1
    geotiff_ds['Sentinel1_VV'].attrs['wavelength'] = '5.55 cm'
    geotiff_ds['Sentinel1_VV'].attrs['missing_value'] = np.nan

    geotiff_ds['Sentinel1_VH'].attrs['standard_name'] = 'surface_backwards_scattering_coefficient_of_radar_wave'
    geotiff_ds['Sentinel1_VH'].attrs['long_name'] = 'SAR_VH_polarized_backscatter'
    geotiff_ds['Sentinel1_VH'].attrs['units'] = 'dB'
    geotiff_ds['Sentinel1_VH'].attrs['valid_max'] = 1
    geotiff_ds['Sentinel1_VH'].attrs['wavelength'] = '5.55 cm'
    geotiff_ds['Sentinel1_VH'].attrs['missing_value'] = np.nan

    geotiff_ds['Sentinel1_HH'].attrs['standard_name'] = 'surface_backwards_scattering_coefficient_of_radar_wave'
    geotiff_ds['Sentinel1_HH'].attrs['long_name'] = 'SAR_HH_polarized_backscatter'
    geotiff_ds['Sentinel1_HH'].attrs['units'] = 'dB'
    geotiff_ds['Sentinel1_HH'].attrs['valid_max'] = 1
    geotiff_ds['Sentinel1_HH'].attrs['wavelength'] = '5.55 cm'
    geotiff_ds['Sentinel1_HH'].attrs['missing_value'] = np.nan

    geotiff_ds['Sentinel1_HV'].attrs['standard_name'] = 'surface_backwards_scattering_coefficient_of_radar_wave'
    geotiff_ds['Sentinel1_HV'].attrs['long_name'] = 'SAR_HV_polarized_backscatter'
    geotiff_ds['Sentinel1_HV'].attrs['units'] = 'dB'
    geotiff_ds['Sentinel1_HV'].attrs['valid_max'] = 1
    geotiff_ds['Sentinel1_HV'].attrs['wavelength'] = '5.55 cm'
    geotiff_ds['Sentinel1_HV'].attrs['missing_value'] = np.nan

    geotiff_ds['Sentinel1_incidence_angle'].attrs['standard_name'] = 'angle_of_incidence'
    geotiff_ds['Sentinel1_incidence_angle'].attrs['units'] = 'degree'
    geotiff_ds['Sentinel1_incidence_angle'].attrs['valid_min'] = 0
    geotiff_ds['Sentinel1_incidence_angle'].attrs['valid_max'] = 90
    geotiff_ds['Sentinel1_incidence_angle'].attrs['missing_value'] = np.nan

    #########################################################################################################
    # Optional filtering by lat/lon
    if coord_bounds:
        geotiff_ds = geotiff_ds.sel(longitude=slice(coord_bounds[0], coord_bounds[1])).sel(latitude=slice(coord_bounds[3], coord_bounds[2]))

    #########################################################################################################
    return geotiff_ds

In [ ]:
# Test out the processing function
dt = datetime.datetime.fromtimestamp(times_Landsat[0]/1000, datetime.UTC).strftime('%Y-%m-%dT%H:%M:%SZ')
ds = process_Landsat_Sentinel_tif(Landsat_tif_list[0], dt)
ds

In [ ]:
# Runs processing function on all the geotif files for each city
for city in cities:
    processed_dir = '/glade/derecho/scratch/jstarfeldt/uhminicubes/hresgrid'
    times_Landsat = pd.read_csv(f'{tmp_dataset_root}/Landsat_times/Landsat_times_{city}.csv').value
    Landsat_root = f'{dataset_root}/{city}/Landsat'
    Landsat_tif_list = glob.glob(Landsat_root+'/Landsat8*.tif')
    Landsat_tif_list = sorted(Landsat_tif_list, key=sort_func_Landsat)

    for i in range(len(Landsat_tif_list)):
        dt_str = Landsat_tif_list[i].split('_')[-1].split('.')[0]
        fname = f'hresgrid_{city}_{dt_str}.nc'
        dt = datetime.datetime.fromtimestamp(times_Landsat[i]/1000, datetime.UTC)
        dt_str = dt.strftime('%Y-%m-%dT%H:%M:%SZ')
        ds = process_Landsat_Sentinel_tif(Landsat_tif_list[i], dt_str)
        ds.to_netcdf(f'{processed_dir}/{str(dt.year)}/{stringify(dt.month)}/{city}/{fname}')